# Chapter 3 — AFM Instrumentation: Python Exercises (Colab)

This notebook accompanies **Chapter 3 (AFM Instrumentation & Detection Systems)**.

It contains three short computational exercises designed to make key instrumentation concepts tangible:

1. **Cantilever force–deflection (Hooke’s law)**
2. **Optical lever amplification (beam deflection detection)**
3. **Thermal noise of a cantilever (equipartition theorem)**

Each exercise includes:
- a short problem statement,
- runnable code,
- plots,
- and (where useful) interactive sliders.

---
## Colab setup
Colab supports interactive widgets via `ipywidgets`. Run the next cell once.


In [ ]:
# Install + enable ipywidgets in Google Colab
!pip -q install ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

# Matplotlib defaults (do not set specific colors)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True


---
# Exercise 1 — Cantilever Force–Deflection Simulation (Hooke’s law)

### Problem statement
An AFM cantilever can be approximated as a linear spring:

$$F = k\,\Delta z$$

1. Choose a cantilever spring constant \(k\) (in N/m).
2. Generate deflections between −20 nm and +20 nm.
3. Compute the corresponding force.
4. Plot **force vs. deflection**.

Use the slider to explore how the stiffness changes the slope.


In [ ]:
def hookes_law_plot(k_N_per_m=0.1, z_min_nm=-20.0, z_max_nm=20.0, n_points=400):
    z_nm = np.linspace(z_min_nm, z_max_nm, int(n_points))
    z_m = z_nm * 1e-9
    F_N = k_N_per_m * z_m

    plt.figure(figsize=(6.2, 4.0))
    plt.plot(z_nm, F_N)
    plt.xlabel('Deflection Δz (nm)')
    plt.ylabel('Force F (N)')
    plt.title('Hooke’s law: Force vs Deflection')
    plt.show()

interact(
    hookes_law_plot,
    k_N_per_m=FloatSlider(value=0.1, min=0.005, max=100.0, step=0.005, description='k (N/m)', readout_format='.3f'),
    z_min_nm=FloatSlider(value=-20.0, min=-200.0, max=-1.0, step=1.0, description='z min (nm)'),
    z_max_nm=FloatSlider(value=20.0, min=1.0, max=200.0, step=1.0, description='z max (nm)'),
    n_points=IntSlider(value=400, min=100, max=2000, step=100, description='points')
);


### Quick check (optional)
Compute a single force value for a given deflection.


In [ ]:
def force_from_deflection(k_N_per_m=0.1, deflection_nm=5.0):
    F_N = k_N_per_m * (deflection_nm * 1e-9)
    print(f"k = {k_N_per_m:.3g} N/m, Δz = {deflection_nm:.3g} nm → F = {F_N:.3g} N")
    print(f"(= {F_N*1e12:.3g} pN)")

interact(
    force_from_deflection,
    k_N_per_m=FloatSlider(value=0.1, min=0.005, max=100.0, step=0.005, description='k (N/m)', readout_format='.3f'),
    deflection_nm=FloatSlider(value=5.0, min=-200.0, max=200.0, step=0.5, description='Δz (nm)')
);


---
# Exercise 2 — Optical Lever Amplification

### Problem statement
In optical beam deflection detection, a small cantilever tilt angle \(\theta\) leads to a laser spot displacement on the photodiode:

$$\Delta x \approx 2L\theta$$

1. Define tilt angles between \(10^{-7}\) and \(10^{-5}\) rad.
2. Compute \(\Delta x\) for detector distances \(L\) of 2 cm, 5 cm, and 10 cm.
3. Plot \(\Delta x\) vs \(\theta\).

Use the interactive plot to explore how increasing \(L\) increases sensitivity.


In [ ]:
def optical_lever_plot(L_cm=5.0, theta_min=1e-7, theta_max=1e-5, n_points=300, y_unit='µm'):
    L_m = L_cm * 1e-2
    theta = np.logspace(np.log10(theta_min), np.log10(theta_max), int(n_points))
    dx_m = 2 * L_m * theta

    if y_unit == 'nm':
        y = dx_m * 1e9
        ylabel = 'Spot displacement Δx (nm)'
    elif y_unit == 'µm':
        y = dx_m * 1e6
        ylabel = 'Spot displacement Δx (µm)'
    else:
        y = dx_m
        ylabel = 'Spot displacement Δx (m)'

    plt.figure(figsize=(6.2, 4.0))
    plt.loglog(theta, y)
    plt.xlabel('Cantilever tilt θ (rad)')
    plt.ylabel(ylabel)
    plt.title('Optical lever amplification: Δx ≈ 2Lθ')
    plt.show()

interact(
    optical_lever_plot,
    L_cm=FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='L (cm)'),
    theta_min=FloatSlider(value=1e-7, min=1e-9, max=1e-6, step=1e-9, description='θ min', readout_format='.1e'),
    theta_max=FloatSlider(value=1e-5, min=1e-6, max=1e-3, step=1e-6, description='θ max', readout_format='.1e'),
    n_points=IntSlider(value=300, min=100, max=2000, step=100, description='points'),
    y_unit=Dropdown(options=['nm','µm','m'], value='µm', description='Δx unit')
);


### Fixed comparison plot (2 cm, 5 cm, 10 cm)
This cell reproduces the exercise request directly.


In [ ]:
theta = np.logspace(-7, -5, 400)
Ls_cm = [2, 5, 10]

plt.figure(figsize=(6.2, 4.0))
for L_cm in Ls_cm:
    dx_um = (2 * (L_cm*1e-2) * theta) * 1e6
    plt.loglog(theta, dx_um, label=f"L = {L_cm} cm")

plt.xlabel('Cantilever tilt θ (rad)')
plt.ylabel('Spot displacement Δx (µm)')
plt.title('Optical lever: effect of detector distance')
plt.legend()
plt.show()


---
# Exercise 3 — Thermal Noise of an AFM Cantilever

### Problem statement
Thermal motion sets a fundamental limit to AFM sensitivity. For a cantilever approximated as a spring:

$$\langle z^2 \rangle = \frac{k_B T}{k}$$

1. Choose a cantilever spring constant \(k\).
2. Compute the root-mean-square (RMS) thermal deflection \(z_{\mathrm{rms}}=\sqrt{\langle z^2\rangle}\).
3. Simulate thermal fluctuations as Gaussian noise with variance \(\sigma^2 = k_B T/k\).
4. Plot a histogram of the simulated displacement.

Use the sliders to see how stiffness and temperature change the thermal noise.


In [ ]:
kB = 1.380649e-23  # J/K

def thermal_noise_sim(k_N_per_m=0.2, T_K=300.0, n_samples=20000, bins=60):
    # Theoretical RMS
    z2 = kB * T_K / k_N_per_m
    z_rms_m = np.sqrt(z2)

    # Simulated displacements (Gaussian assumption)
    z = np.random.normal(loc=0.0, scale=z_rms_m, size=int(n_samples))

    # Plot histogram in nm
    plt.figure(figsize=(6.2, 4.0))
    plt.hist(z * 1e9, bins=int(bins), density=True)
    plt.xlabel('Thermal deflection z (nm)')
    plt.ylabel('Probability density')
    plt.title('Thermal noise distribution (Gaussian model)')
    plt.show()

    print(f"k = {k_N_per_m:.4g} N/m, T = {T_K:.1f} K")
    print(f"Theoretical z_rms = {z_rms_m*1e9:.4g} nm")

interact(
    thermal_noise_sim,
    k_N_per_m=FloatSlider(value=0.2, min=0.005, max=100.0, step=0.005, description='k (N/m)', readout_format='.3f'),
    T_K=FloatSlider(value=300.0, min=250.0, max=330.0, step=1.0, description='T (K)'),
    n_samples=IntSlider(value=20000, min=2000, max=200000, step=2000, description='samples'),
    bins=IntSlider(value=60, min=20, max=200, step=5, description='bins')
);


### Optional extension — Compare two cantilevers
This cell compares the theoretical RMS thermal deflection for two different spring constants.


In [ ]:
def compare_two_cantilevers(k1=0.03, k2=40.0, T_K=300.0):
    z1 = np.sqrt(kB*T_K/k1) * 1e9
    z2 = np.sqrt(kB*T_K/k2) * 1e9
    print(f"T = {T_K:.1f} K")
    print(f"k1 = {k1:.3g} N/m → z_rms ≈ {z1:.4g} nm")
    print(f"k2 = {k2:.3g} N/m → z_rms ≈ {z2:.4g} nm")

interact(
    compare_two_cantilevers,
    k1=FloatSlider(value=0.03, min=0.005, max=5.0, step=0.005, description='k1 (N/m)', readout_format='.3f'),
    k2=FloatSlider(value=40.0, min=0.1, max=200.0, step=0.1, description='k2 (N/m)', readout_format='.1f'),
    T_K=FloatSlider(value=300.0, min=250.0, max=330.0, step=1.0, description='T (K)')
);


---
# Wrap-up

You have now:
- simulated the linear force transduction of an AFM cantilever,
- explored how the optical lever converts tiny angles into measurable beam displacement,
- and quantified the fundamental thermal noise limit.

For report-style answers, students can export figures and briefly discuss:
- how **k** affects force sensitivity and thermal noise,
- how **L** affects optical detection sensitivity,
- and how these trade-offs influence AFM operation on soft biological samples.
